In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(
    "/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env"
)

# api_key = os.getenv("CO_API_KEY")
# print(api_key)

load_dotenv(dotenv_path=os.path.join("..", ".env"))

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY not found. Add it to the project-root .env file."
)

# The client reads OPENAI_API_KEY from the environment automatically.
client = OpenAI()

MODEL = "gpt-6-luna"  # strong general/coding model; or "gpt-6-luna" for fast, high-volume tasks
print(f"Client ready. Using model: {MODEL}")

Client ready. Using model: gpt-6-luna


# OpenAI Prompt Engineering Patterns (Responses API)

This notebook demonstrates common prompt types and patterns using the OpenAI Python SDK (`client.responses.create` and `client.responses.stream`).

## 1. Zero-Shot Prompting
Asking the model to perform a task directly without giving any prior examples.

In [4]:
response = client.responses.create(
    model=MODEL,
    max_output_tokens=200,
    input=(
        "Classify the sentiment of this review as Positive, Neutral, or Negative:\n\n"
        "'The packaging was damaged, though the product inside works is also damaged. I am really disappointed with the quality and service. I expected better.'"
    ),
)

print(response.output_text)

Negative.


## 2. System / Instruction Prompting (Role & Constraints)
Using `instructions` to define persona, tone, and formatting constraints.

In [ ]:
response = client.responses.create(
    model=MODEL,
    max_output_tokens=250,
    instructions=(
        "You are a  maths researcher. Please answer the following question in brief and provide examples."
    ),
    input="How does the Pythagorean theorem work? Can you provide a proof and an example of its application?",
)

print(response.output_text)

## 3. Few-Shot Prompting
Providing input-output examples inside the prompt so the model follows an exact structure and pattern.

In [ ]:
few_shot_prompt = """Extract the drug name and dosage frequency from each clinical note.

Note: Patient started on Lisinopril 10mg once daily for hypertension.
Output: {"drug": "Lisinopril", "frequency": "once daily"}

Note: Prescribed Metformin 500mg twice a day with meals.
Output: {"drug": "Metformin", "frequency": "twice a day"}

Note: Administer Amoxicillin 250mg every 8 hours for 7 days.
Output:"""

response = client.responses.create(
    model=MODEL,
    max_output_tokens=150,
    input=few_shot_prompt,
)

print(response.output_text)

## 4. Chain-of-Thought (CoT) Prompting
Guiding the model to break down reasoning step-by-step before giving the final answer.

In [ ]:
cot_prompt = """A clinical trial enrolls 240 patients across 3 sites.
Site A enrolls twice as many patients as Site B.
Site C enrolls 40 fewer patients than Site A.
How many patients are enrolled at each site?

Think through the steps carefully before stating the final numbers."""

response = client.responses.create(
    model=MODEL,
    max_output_tokens=400,
    input=cot_prompt,
)

print(response.output_text)

## 5. Structured Output Prompting (JSON Schema)
Enforcing a strict JSON schema with `text={'format': ...}` so responses parse cleanly into Python dicts.

In [ ]:
import json

response = client.responses.create(
    model=MODEL,
    input="Summarize the key details of ICH E2A in pharmaceutical regulation.",
    text={
        "format": {
            "type": "json_schema",
            "name": "ich_guideline_summary",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {
                    "code": {"type": "string"},
                    "title": {"type": "string"},
                    "purpose": {"type": "string"},
                    "key_definitions": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
                "required": ["code", "title", "purpose", "key_definitions"],
                "additionalProperties": False,
            },
        }
    },
)

parsed = json.loads(response.output_text)
print(json.dumps(parsed, indent=2))

## 6. Context-Grounded Prompting with Delimiters (RAG Style)
Using XML tags to separate instructions from reference text and prevent hallucination.

In [ ]:
sop_context = """
SOP-QA-014 (Effective Date: 2026-03-01):
All temperature excursions in cold-chain storage (2°C to 8°C) exceeding 30 minutes
must be reported to Quality Assurance within 2 hours of detection and quarantined
pending stability review.
"""

prompt = f"""Answer the question using ONLY the provided SOP context below.
If the answer is not present in the context, reply with "Information not found in SOP."

<context>
{sop_context}
</context>

Question: Within how many hours must a 45-minute cold-chain temperature excursion be reported to QA?"""

response = client.responses.create(
    model=MODEL,
    max_output_tokens=150,
    input=prompt,
)

print(response.output_text)

## 7. Multi-Turn Conversational Prompting
Maintaining context across turns by passing conversation history (or chaining via `previous_response_id`).

In [ ]:
conversation = [
    {"role": "user", "content": "What are the 3 main pre-approval clinical trial phases?"},
]

res1 = client.responses.create(
    model=MODEL,
    max_output_tokens=250,
    input=conversation,
)
print("Turn 1:\n", res1.output_text)

# Append assistant reply and ask a follow-up question
conversation.append({"role": "assistant", "content": res1.output_text})
conversation.append({"role": "user", "content": "Which phase typically involves the largest number of participants?"})

res2 = client.responses.create(
    model=MODEL,
    max_output_tokens=150,
    input=conversation,
)
print("\nTurn 2:\n", res2.output_text)

## 8. Prompt Chaining (Multi-Step Pipeline)
Feeding the output of one prompt into the next prompt to build reliable multi-stage workflows.

In [ ]:
raw_notes = """
Meeting notes: Batch #402 showed higher moisture content (4.8% vs 4.5% limit).
Operator noted the dryer inlet temp fluctuated between 62C and 68C instead of steady 65C.
Need to recalibrate thermocouple TC-04 and re-test retained samples before batch release.
"""

# Step 1: Extract root cause and actions
step1 = client.responses.create(
    model=MODEL,
    max_output_tokens=200,
    input=f"Extract the root cause and required corrective actions from these notes:\n{raw_notes}",
)
findings = step1.output_text
print("--- Step 1: Extracted Findings ---")
print(findings)

# Step 2: Transform into a formal CAPA Markdown table
step2 = client.responses.create(
    model=MODEL,
    max_output_tokens=250,
    instructions="Convert the findings into a formal CAPA summary table in Markdown.",
    input=findings,
)
print("\n--- Step 2: Formal CAPA Table ---")
print(step2.output_text)

## 9. Streaming Prompt Responses
Streaming text chunks in real time using `client.responses.stream`.

In [ ]:
with client.responses.stream(
    model=MODEL,
    max_output_tokens=256,
    input="Explain what a 483 observation is in three concise sentences.",
) as stream:
    for event in stream:
        if event.type == "response.output_text.delta":
            print(event.delta, end="", flush=True)